In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import (
    ttest_1samp, ttest_ind, ttest_rel, f_oneway,
    mannwhitneyu, wilcoxon, kruskal, spearmanr, kendalltau,
    chi2_contingency, shapiro, ks_2samp, normaltest
)
import warnings
warnings.filterwarnings('ignore')

In [3]:
%pip install matplotlib seaborn scipy pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
df_raw = pd.read_csv('twitter-posts.csv')

In [14]:
print(f"Dimensiones: {df_raw.shape}")
print(f"Memoria: {df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Columnas: {list(df_raw.columns)}")

print("\n3.2. TIPOS DE DATOS Y VALORES NULOS")
print("-"*50)
null_info = pd.DataFrame({
    'Tipo': df_raw.dtypes,
    'Nulos': df_raw.isnull().sum(),
    '% Nulos': (df_raw.isnull().sum() / len(df_raw) * 100).round(2),
    'Únicos': df_raw.nunique()
})
print(null_info[null_info['Nulos'] > 0].sort_values('% Nulos', ascending=False).head(15))

print("\n3.3. ESTADÍSTICAS DESCRIPTIVAS BÁSICAS")
print("-"*50)
print(df_raw.describe().round(2))

print("\n3.4. VALORES EXTREMOS Y OUTLIERS (Método IQR)")
print("-"*50)
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df_raw[col].notna().sum() > 0:
        Q1 = df_raw[col].quantile(0.25)
        Q3 = df_raw[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers = ((df_raw[col] < (Q1 - 1.5 * IQR)) | (df_raw[col] > (Q3 + 1.5 * IQR))).sum()
        if outliers > 0:
            print(f"  {col}: {outliers} outliers ({outliers/len(df_raw)*100:.1f} %)")

print("\n3.5. VALORES DUPLICADOS")
print("-"*50)
print(f"Duplicados exactos (todas las columnas): {df_raw.duplicated().sum()}")
print(f"Duplicados por ID: {df_raw['id'].duplicated().sum() if 'id' in df_raw.columns else 'N/A'}")

print("\n3.6. FORMATOS INCONSISTENTES (Ejemplos)")
print("-"*50)
for col in df_raw.select_dtypes(include=['object']).columns:
    unique_vals = df_raw[col].dropna().unique()[:5]
    print(f"  {col}: {unique_vals} ...")

Dimensiones: (1000, 25)
Memoria: 1.99 MB
Columnas: ['id', 'user_posted', 'name', 'description', 'date_posted', 'photos', 'videos', 'url', 'quoted_post', 'tagged_users', 'replies', 'reposts', 'likes', 'views', 'external_url', 'hashtags', 'followers', 'biography', 'posts_count', 'profile_image_link', 'following', 'is_verified', 'quotes', 'bookmarks', 'parent_post_details']

📌 3.2. TIPOS DE DATOS Y VALORES NULOS
--------------------------------------------------
                 Tipo  Nulos  % Nulos  Únicos
videos            str    814     81.4     186
external_url      str    780     78.0     218
hashtags          str    767     76.7     225
tagged_users      str    561     56.1     424
photos            str    453     45.3     547
views         float64      5      0.5     986
biography         str      3      0.3     845

📌 3.3. ESTADÍSTICAS DESCRIPTIVAS BÁSICAS
--------------------------------------------------
                 id  replies   reposts      likes        views   followers 

Unas conclusiones que podemos sacar a rapida vista aca: 

1) vemos muchos nulos en algunas categorias. Esto no nos interesa ya que no todos los posts deben contener esa informacion.

2) Se calcula las estadisticas descriptivas basicas para el id y sus correspondientes outliers. al ser el ID un identificador, no nos interesan estos datos ya que no es una variable numerica real que indique algo.

3) la mediana y la media difieren por mucho. asi que eso nos puede explicar que hay pocos tuits virales que elevan el promedio


In [ ]:
df_clean = df_raw.copy()


# 4.1 Manejo de duplicados
print("\n4.1. Eliminando duplicados...")
initial_rows = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"   Duplicados eliminados: {initial_rows - len(df_clean)}")
# Duplicados por ID de post (no debería haber posts repetidos)
if 'id' in df_clean.columns:
    df_clean = df_clean.drop_duplicates(subset='id')

# 4.2 Limpieza de columnas de tiempo
print("\n4.2. Transformando columnas de tiempo...")
if 'date_posted' in df_clean.columns:
    df_clean['date_posted'] = pd.to_datetime(df_clean['date_posted'], errors='coerce', utc=True)
    df_clean['Year'] = df_clean['date_posted'].dt.year
    df_clean['Month'] = df_clean['date_posted'].dt.month
    df_clean['Hour'] = df_clean['date_posted'].dt.hour
    df_clean['DayOfWeek'] = df_clean['date_posted'].dt.dayofweek

# 4.3 Conversión de tipos (posts_count quedó como texto por los "null")
print("\n4.3. Corrigiendo tipos numéricos...")
for col in ['posts_count', 'followers', 'following', 'replies', 'reposts',
            'likes', 'views', 'quotes', 'bookmarks']:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# 4.4 Manejo de valores nulos (estratégico)
print("\n4.4. Imputación de valores nulos...")
print(f"   Nulos antes: {df_clean.isnull().sum().sum()}")

numeric_cols_clean = df_clean.select_dtypes(include=[np.number]).columns
for col in numeric_cols_clean:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print(f"   Nulos después (numéricos): {df_clean.select_dtypes(include=[np.number]).isnull().sum().sum()}")

# 4.5 Tratamiento de outliers extremos (recorte por percentiles 1-99)
print("\n4.5. Tratamiento de outliers extremos...")
before_rows = len(df_clean)
for col in ['likes', 'views', 'reposts', 'replies', 'followers']:
    if col in df_clean.columns and df_clean[col].notna().sum() > 0:
        low = df_clean[col].quantile(0.01)
        high = df_clean[col].quantile(0.99)
        df_clean = df_clean[(df_clean[col] >= low) & (df_clean[col] <= high)]
print(f"   Filas eliminadas: {before_rows - len(df_clean)} ({((before_rows - len(df_clean))/before_rows)*100:.1f} %)")



print(f"\nETL completado. Dataset final: {df_clean.shape[0]} filas, {df_clean.shape[1]} columnas")


📌 4.1. Eliminando duplicados...
   Duplicados eliminados: 0

📌 4.2. Transformando columnas de tiempo...

📌 4.3. Corrigiendo tipos numéricos...

📌 4.4. Imputación de valores nulos...
   Nulos antes: 8383
   Nulos después (numéricos): 4000

📌 4.5. Tratamiento de outliers extremos...
   Filas eliminadas: 82 (8.2 %)

📌 4.6. Creando variables derivadas...

✅ ETL completado. Dataset final: 918 filas, 32 columnas


Luego de este paso verificamos que cada post es unico, por no haber eliminado duplicados

transformamos todas las columnas de tiempo, convertimos a numerico algunas variables que tenian metricas.

unicamente tratamos los nulos de variables numericas, por eso nos quedan 4000 nulos sin trabajar ya que se tratan de variables de texto 

luego solo eliminamos valores extremos de outliers y no por el metodo IQR, ya que sino hubieramos eliminado un 15% de los datos como vimos en el punto 3.4. con esto solo eliminamos los extremos



### Pregunta 1: ¿La cantidad de respuestas (replies) difiere según el número de usuarios etiquetados (tagged_users)?

La palabra clave "difiere" nos indica que debemos comparar grupos. Primero, convertiremos la columna `tagged_users` (JSON) en un conteo numérico. Luego, agruparemos esos conteos en categorías (ej: 0 etiquetas, 1 etiqueta, 2 etiquetas, 3 o más). 
Finalmente, como los datos de respuestas (`replies`) presentan una distribución asimétrica con outliers, aplicaremos la prueba no paramétrica de **Kruskal-Wallis** para comparar si las medianas de respuestas difieren entre estos grupos.

In [11]:
import json
import pandas as pd
from scipy.stats import kruskal

# 1. Transformar JSON a número (igual que antes)
def contar_etiquetas(x):
    if pd.isna(x): return 0
    try: return len(json.loads(x))
    except: return 0

df_clean['num_tagged_users'] = df_clean['tagged_users'].apply(contar_etiquetas)

# 2. Crear los grupos para poder compararlos ("difiere entre grupos")
def agrupar_etiquetas(n):
    if n == 0: return '0 etiquetas'
    elif n == 1: return '1 etiqueta'
    elif n == 2: return '2 etiquetas'
    else: return '3 o más etiquetas'

df_clean['grupo_etiquetas'] = df_clean['num_tagged_users'].apply(agrupar_etiquetas)

from scipy.stats import shapiro

# Justificación formal de no-normalidad (por eso usamos prueba no paramétrica)
print("Test de normalidad (Shapiro-Wilk) sobre replies en cada grupo:")
for cat in df_clean['grupo_etiquetas'].unique():
    datos = df_clean[df_clean['grupo_etiquetas'] == cat]['replies'].dropna()
    if len(datos) >= 3:
        p = shapiro(datos).pvalue
        print(f"  {cat}: p = {p:.6f} {'(no normal)' if p < 0.05 else '(normal)'}")
print("Como los p-valores son < 0.05, se rechaza normalidad por lo que se justifica Kruskal-Wallis.\n")

print("\n RESULTADO PREGUNTA 1: Diferencia de Replies según cantidad de etiquetados (Kruskal-Wallis)")
print("-" * 80)

# 3. Separar los datos de replies por cada grupo creado
grupos = [df_clean[df_clean['grupo_etiquetas'] == cat]['replies'].dropna() 
          for cat in df_clean['grupo_etiquetas'].unique()]

# 4. Filtrar grupos que tengan al menos 5 datos (condición para Kruskal-Wallis)
grupos_validos = [g for g in grupos if len(g) >= 5]

if len(grupos_validos) >= 2:
    # 5. Aplicar la prueba
    h_stat, kw_p = kruskal(*grupos_validos)
    
    print(f"Estadístico H de Kruskal-Wallis: {h_stat:.4f}")
    print(f"Valor p: {kw_p:.6f}\n")
    
    if kw_p < 0.05:
        print("Conclusión: Rechazamos la hipótesis nula. La cantidad de respuestas (replies) DIFIERE significativamente según el grupo/cantidad de usuarios etiquetados.")
    else:
        print("Conclusión: No rechazamos la hipótesis nula. No hay evidencia suficiente para decir que la cantidad de respuestas difiera según la cantidad de usuarios etiquetados.")
else:
    print("No hay suficientes grupos o datos para realizar la prueba de Kruskal-Wallis.")

Test de normalidad (Shapiro-Wilk) sobre replies en cada grupo:
  1 etiqueta: p = 0.000000 (no normal)
  0 etiquetas: p = 0.000000 (no normal)
  3 o más etiquetas: p = 0.000000 (no normal)
  2 etiquetas: p = 0.000000 (no normal)
Como los p-valores son < 0.05, se rechaza normalidad → se justifica Kruskal-Wallis.


 RESULTADO PREGUNTA 1: Diferencia de Replies según cantidad de etiquetados (Kruskal-Wallis)
--------------------------------------------------------------------------------
Estadístico H de Kruskal-Wallis: 17.3813
Valor p: 0.000590

Conclusión: Rechazamos la hipótesis nula. La cantidad de respuestas (replies) DIFIERE significativamente según el grupo/cantidad de usuarios etiquetados.


Con este resultado podemos decir que hay evidencia estadística que la cantidad de respuestas  cambia según cuántos usuarios se etiquetan en el post.


### Pregunta 2: ¿La cantidad de posts del usuario (posts_count) está asociada con el número de likes que recibe un post?
Para esta segunda consigna, las variables `posts_count` y `likes` ya son numéricas, por lo tanto aplicamos el test de shapiro para verificar si se trataba de una muestra parametrica o no parametrica. y luego aplicamos correlacion de Spearman para corroborar si subir mas posteos esta relacionado a generar mas likes

In [10]:
print("\n RESULTADO PREGUNTA 2: Asociación entre Posts Count y Likes")
print("-" * 65)


temp_q2 = df_clean[['posts_count', 'likes']].dropna()

# Justificación formal de no-normalidad
print("Test de normalidad (Shapiro-Wilk):")
print(f"  posts_count: p = {shapiro(temp_q2['posts_count']).pvalue:.6f}")
print(f"  likes: p = {shapiro(temp_q2['likes']).pvalue:.6f}")
print("  (p < 0.05, no normales, se justifica Spearman sobre Pearson)\n")

if len(temp_q2) > 1:

    rho_q2, p_val_q2 = spearmanr(temp_q2['posts_count'], temp_q2['likes'])
    
    print(f"Rho de Spearman: {rho_q2:.4f}")
    print(f"Valor p: {p_val_q2:.6f}\n")
    
    if p_val_q2 < 0.05:
        direccion = 'positiva' if rho_q2 > 0 else 'negativa'
        print(f"Conclusión: Rechazamos la hipótesis nula. Existe una asociación significativa ({direccion}) entre la cantidad histórica de posts del usuario y los likes que recibe en un post.")
    else:
        print("Conclusión: No rechazamos la hipótesis nula. No hay evidencia estadística suficiente que asocie la cantidad de posts del usuario con los likes recibidos.")
else:
    print("No hay suficientes datos válidos para realizar la prueba.")


 RESULTADO PREGUNTA 2: Asociación entre Posts Count y Likes
-----------------------------------------------------------------
Test de normalidad (Shapiro-Wilk):
  posts_count: p = 0.000000
  likes: p = 0.000000
  (p < 0.05 → no normales → se justifica Spearman sobre Pearson)

Rho de Spearman: 0.2223
Valor p: 0.000000

Conclusión: Rechazamos la hipótesis nula. Existe una asociación significativa (positiva) entre la cantidad histórica de posts del usuario y los likes que recibe en un post.


dado que el p-valor < 0.05 entendemos que la relacion existe y no fue por azar.
Sin embargo, como el rho < a 0.3 (0.22), esa relación es débil. Podemos afirmar que hay una tendencia a que los usuarios más activos reciban más likes, pero es leve.